# ⚡ Retune Siêu Tốc (Nhánh Test)
**Mục tiêu:** Tinh chỉnh Threshold nhanh gọn từ các file kết quả cũ mà không cần chạy lại Retrieval.

### 📁 File cần có trong Drive > `R2AI_Artifacts`:
- `retrieval.db` (Database chứa điểm RRF và Candidates) ✅
- `results_partial.jsonl` (File backup generation nếu có) ✅
- `R2AIStage1DATA.json` (Đề bài) ✅

## Cell 1 — Mount Drive & Chuẩn bị

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f'\n📋 Scan {DRIVE_DIR}:')
for root, dirs, files in os.walk(DRIVE_DIR):
    level = root.replace(DRIVE_DIR, '').count(os.sep)
    indent = '  ' * level
    rel = root.replace(DRIVE_DIR, '') or '/'
    print(f'{indent}📁 {rel}/')
    for f in files:
        sz = os.path.getsize(os.path.join(root, f)) / 1024 / 1024
        print(f'{indent}  📄 {f} ({sz:.1f}MB)')
print('\n✅ Cell 1 Done!')


## Cell 2 — Git Clone & Restore Artifacts

In [ ]:
import os, sys, shutil, subprocess
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'
WORK_DIR  = '/content/sme-legal-assistant'
REPO_URL  = 'https://github.com/Platypus27-coder/sme-legal-assistant.git'
BRANCH    = 'test'

if os.path.exists(WORK_DIR): shutil.rmtree(WORK_DIR)

print(f'⬇️ Clone branch {BRANCH} từ Github...')
r = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, WORK_DIR], capture_output=True, text=True)
if r.returncode != 0:
    print(f'❌ Lỗi Clone: {r.stderr}')
    raise SystemExit

# Tạo thư mục
for d in ['output', 'cache', 'raw', 'index']:
    os.makedirs(f'{WORK_DIR}/artifacts/{d}', exist_ok=True)
os.makedirs(f'{WORK_DIR}/data', exist_ok=True)

# Copy files từ Drive vào Project
print('📦 Restore Artifacts từ Drive...')
files_to_restore = [
    ('retrieval.db', 'artifacts/cache/retrieval.db'),
    ('results_partial.jsonl', 'artifacts/output/results_partial.jsonl'),
    ('R2AIStage1DATA.json', 'data/R2AIStage1DATA.json')
]

for src_name, dst_path in files_to_restore:
    src_full = f'{DRIVE_DIR}/{src_name}'
    if os.path.exists(src_full):
        shutil.copy2(src_full, f'{WORK_DIR}/{dst_path}')
        print(f'  ✅ Restored: {src_name}')
    else:
        print(f'  ⚠️ Missing: {src_name} (Thực thi có thể lỗi nếu thiếu file này)')

sys.path.insert(0, f'{WORK_DIR}/src')
os.chdir(WORK_DIR)
print('\n✅ Cell 2 Done!')


## Cell 3 — Cài đặt thư viện nhẹ

In [ ]:
import subprocess, sys
print('📦 Cài đặt thư viện...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers>=2.7.0'], capture_output=True, text=True)
print('✅ Cell 3 Done!')


## Cell 4 — Tinh chỉnh Threshold (Retune)

In [ ]:
import subprocess

print('🚀 Bắt đầu chấm điểm lại (Retune)...')
# Script rerank_retune.py sẽ đọc file retrieval.db và results_partial.jsonl để xuất ra ZIP
# Mở file rerank_retune.py nếu bạn muốn thay đổi DEFAULT_HIGH_CONF (mặc định 0.62)
r = subprocess.run(['python', 'rerank_retune.py'], capture_output=False)

print('\n🎉 HOÀN TẤT! File submission_reranked.zip đã được lưu vào Google Drive của bạn.')
